## The first big project - Professionally You!

### And, Tool use.

### But first: introducing Mailjet

Mailjet is a nifty tool for sending emails.

It's super easy to set up and install!

Simply visit https://github.com/mailjet/mailjet-apiv3-python and Follow the instructions.

In [1]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr
from mailjet_rest import Client
from pydantic import BaseModel
from agents import Agent, Runner, trace, function_tool

In [2]:
# The usual start

load_dotenv(override=True)
openai = OpenAI()

In [3]:
# For pushover

mailjet_key = os.getenv("MAILJET_API_KEY")
mailjet_secret = os.getenv("MAILJET_API_SECRET")
mailjet = Client(auth=(mailjet_key, mailjet_secret))

if mailjet_key:
    print(f"Mailjet API key found and starts with {mailjet_key[0]}")
else:
    print("Mailjet API key not found")
if mailjet_secret:
    print(f"Mailjet secret found and starts with {mailjet_secret[0]}")
else:
    print("Mailjet secret not found")

Mailjet API key found and starts with d
Mailjet secret found and starts with f


In [4]:
@function_tool
def send_email(email: str, notes: str = "") -> dict:
    data = {
        "FromEmail": "imtiaz.emu.it@gmail.com",
        "FromName": "Md Imtiaz Hossain",
        "Subject": f"The person with email {email} wants to connect!",
        "Text-part": notes,
        "Html-part": f'<h3>{notes}</h3>',
        "Recipients": [{"Email": "imtiaz.emu.it2@gmail.com"}],
    }
    
    mailjet.send.create(data=data)
    return {"status": "Email sent successfully"}
    

In [10]:
# Create main agent with tools
name = "Md Imtiaz Hossain Emu"

main_agent = Agent(
    name=name,
    instructions=f"""You are acting as {name}. You are answering questions on your website, 
    particularly questions related to your career, background, skills and experience. 
    Your responsibility is to represent yourself for interactions on the website as faithfully as possible. 
    Be professional and engaging, as if talking to a potential client or future employer who came across the website. 
    If you don't know the answer to any question, acknowledge it politely.
    If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email.
    Once you have the user's email, use the send_email tool to notify yourself about the interested user.
    while calling the tool, pass the user question that you've failed to answer along with the email user left.
    Always be polite and professional. Never reveal that you are an AI or a bot. 
    Always stay in character.""",
    model="gpt-4o-mini",
    tools=[send_email],
)

In [6]:
reader = PdfReader("me/Resume_Md_Imtiaz_Hossain.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [11]:
# Add context to the agent
context_prompt = f"""
## Summary:
{summary}

## LinkedIn Profile/Resume:
{linkedin}

With this context, please chat with the user, always staying in character as {name}.
"""

# Update the agent with context
main_agent.instructions += context_prompt

In [12]:
async def chat(message, history):
    # Convert Gradio history format to messages format
    messages = []
    for msg in history:
        messages.append({"role": msg["role"], "content": msg["content"]})
    
    # Add the new user message
    messages.append({"role": "user", "content": message})
    
    # Create a Runner and run with the agent
    result = await Runner.run(main_agent, messages)
    
    # Return the final response
    return result.final_output

In [13]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
